In [34]:
import pandas as pd

In [35]:
from lingua import Language, LanguageDetectorBuilder

In [36]:
fh = "../data/public/TMS-title-lang.csv"

In [37]:
df = pd.read_csv(fh, names=["objectID", "title", "label", "local_label", "iso"], dtype={"title": str})

C:\Users\tomaszkalata\AppData\Local\Temp\ipykernel_14800\2567249600.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(fh, names=["objectID", "title", "label", "local_label", "iso"], dtype={"title": str})


In [38]:
df.head()

,objectID,title,label,local_label,iso
0,Object_ID,Title,Label,Local_Label,ISO_Code_With_Locale
1,17733,[Negatives and digital scans of Morris Huberla...,English (US),English,en-US
2,17748,Jesse Reads a Book.,English (US),English,en-US
3,17755,The River That Runs Two Ways,English (US),English,en-US
4,17771,Photographs of Native Americans,English (US),English,en-US


In [39]:
unique_lang = df["iso"].unique()

In [40]:
unique_lang.shape[0]

2

In [41]:
unique_lang.tolist()

['ISO_Code_With_Locale', 'en-US']

In [42]:
detector = LanguageDetectorBuilder.from_all_languages().build()

In [53]:
no_title = df[df["title"].isna()]

In [54]:
no_title.shape[0]

2

In [55]:
no_title.head()

,objectID,title,label,local_label,iso
85987,54260,NaN,English (US),English,en-US
181482,117673,NaN,English (US),English,en-US


In [56]:
df.dropna(subset=["title"], inplace=True)

In [59]:
def detect_lang(row: str) -> str:
    language = detector.detect_language_of(row["title"])
    if language:
        confidence = detector.compute_language_confidence(row["title"], language)
        return f"{language.name}, {confidence:.2f}"
    else:
        return "UNKNOWN"

In [60]:
df["detected_lang"] = df.apply(detect_lang, axis=1)

In [61]:
df.head()

,objectID,title,label,local_label,iso,detected_lang
0,Object_ID,Title,Label,Local_Label,ISO_Code_With_Locale,"TURKISH, 0.17"
1,17733,[Negatives and digital scans of Morris Huberla...,English (US),English,en-US,"ENGLISH, 0.32"
2,17748,Jesse Reads a Book.,English (US),English,en-US,"ENGLISH, 0.14"
3,17755,The River That Runs Two Ways,English (US),English,en-US,"ENGLISH, 0.30"
4,17771,Photographs of Native Americans,English (US),English,en-US,"ENGLISH, 0.47"


In [62]:
new_cols = df["detected_lang"].str.split(", ", expand=True)

In [64]:
new_cols.columns = ["lang", "confidence"]

In [69]:
dfp =  pd.concat([df, new_cols], axis=1).drop(columns=["detected_lang", "label", "local_label", "iso"])

In [70]:
dfp.head()

,objectID,title,lang,confidence
0,Object_ID,Title,TURKISH,0.17
1,17733,[Negatives and digital scans of Morris Huberla...,ENGLISH,0.32
2,17748,Jesse Reads a Book.,ENGLISH,0.14
3,17755,The River That Runs Two Ways,ENGLISH,0.30
4,17771,Photographs of Native Americans,ENGLISH,0.47


In [71]:
dfp.to_csv("../data/public/TMS-title-lang-detected.csv", index=False)